In [ ]:
# Installing The Necessary Packages And Dependencies Required For Evaluating The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

!pip install -q transformers datasets torchaudio librosa jiwer evaluate tqdm python-levenshtein editdistance scikit-learn soundfile huggingface_hub

In [ ]:
# Importing The Necessary Packages And Dependencies Required And Loading The Svarah Dataset For Evaluating The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset


import torch
import numpy as np

import editdistance
import evaluate
import jiwer
import librosa

from datasets import Audio, load_dataset
from sklearn.metrics import precision_recall_fscore_support
from transformers import AutoModelForCTC, AutoProcessor

print("The Necessary Packages And Dependencies Required Have Been Successfully Imported")

In [ ]:
# Loading The Fine-Tuned Wav2Vec2 Model And Processor Configuration For Evaluating The Speech Recognition Capabilities

processor = AutoProcessor.from_pretrained("matheetharanadshyan/wav2vec2-svarah")

model = AutoModelForCTC.from_pretrained("matheetharanadshyan/wav2vec2-svarah")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model.eval()

print(f"The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset Has Been Successfully Loaded On : {device}")

In [ ]:
# Loading The Svarah Dataset And Preparing For The Evaluation For Evaluating The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

from datasets import Audio, load_dataset
from huggingface_hub import login

login()

dataset = load_dataset("ai4bharat/Svarah", split="test")
dataset = dataset.cast_column("audio_filepath", Audio(sampling_rate=16000))

print("The Svarah Dataset Has Been Loaded And The Audio Recordings Have Been Resampled To 16KHz")

In [ ]:
# Performing Batch-Based Speech Recognition Inference On The Audio Recordings Using The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

import torch
from tqdm.auto import tqdm

all_predictions = []
all_references = []

BATCH_SIZE = 16

model.eval()
print(f"Starting Batch Inference Across {len(dataset):,} Audio Recording Evaluation Samples")

with torch.no_grad():
    for index in tqdm(range(0, len(dataset), BATCH_SIZE)):
        batch = dataset[index : index + BATCH_SIZE]
        audio_arrays = [sample["array"] for sample in batch["audio_filepath"]]
        
        inputs = processor(audio_arrays, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
        outputs = model(inputs.input_values)

        predicted_ids = torch.argmax(outputs.logits, dim=-1)
        predicted_transcriptions = processor.batch_decode(predicted_ids)

        all_predictions.extend(predicted_transcriptions)
        all_references.extend(batch["text"])

print("")
print(f"The Inference Has Been Completed Successfully Across {len(all_predictions):,} Audio Recording Evaluation Samples")

In [ ]:
# Computing Automatic Speech Recognition Evaluation Metrics And Visualizing The Fine-Tuned Wav2Vec2 Model's Evaluation Results

import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import auc, confusion_matrix, roc_curve

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

clean_predictions = [prediction.strip().lower() for prediction in all_predictions]
clean_references = [reference.strip().lower() for reference in all_references]

wer_score = wer_metric.compute(predictions=clean_predictions, references=clean_references)

def calculate_classification_metrics(predictions, references):
    total_correct = 0
    total_reference_words = 0
    total_predicted_words = 0

    for prediction, reference in zip(predictions, references):
        prediction_words = prediction.split()
        reference_words = reference.split()
        common_words = set(prediction_words) & set(reference_words)

        total_correct += len(common_words)
        total_predicted_words += len(prediction_words)
        total_reference_words += len(reference_words)

    precision = total_correct / total_predicted_words if total_predicted_words > 0 else 0
    recall = total_correct / total_reference_words if total_reference_words > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

precision, recall, f1_score = calculate_classification_metrics(clean_predictions, clean_references)

all_prediction_characters = list("".join(clean_predictions)[:5000])
all_reference_characters = list("".join(clean_references)[:5000])
minimum_length = min(len(all_prediction_characters), len(all_reference_characters))
character_accuracy = sum(1 for index in range(minimum_length) if all_prediction_characters[index] == all_reference_characters[index]) / minimum_length

print("")
print(f"{'The Word Error Rate (WER)':<30} | {wer_score:.4f}")
print(f"{'The Character Error Rate (CER)':<30} | {cer_score:.4f}")
print(f"{'Precision':<30} | {precision:.4f}")
print(f"{'Recall':<30} | {recall:.4f}")
print(f"{'F1-Score':<30} | {f1_score:.4f}")
print(f"{'Character Accuracy':<30} | {character_accuracy:.4f}")

In [ ]:
# Loading And Evaluating The Baseline Wav2Vec2 Model Against The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

import gc
import torch
import evaluate
import numpy as np

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from tqdm.auto import tqdm

try:
    del model
    torch.cuda.empty_cache()
    gc.collect()
except NameError:
    pass

baseline_model_identifier = "facebook/wav2vec2-base-960h"
print(f"Loading The Baseline Wav2Vec2 Model: {baseline_model_identifier}")

baseline_processor = Wav2Vec2Processor.from_pretrained(baseline_model_identifier)
baseline_model = Wav2Vec2ForCTC.from_pretrained(baseline_model_identifier).to(device)

baseline_predictions = []
baseline_references = []
baseline_model.eval()

print("")
print(f"Starting Inference On The Baseline Wav2Vec2 Model Across {len(dataset):,} Audio Recording Evaluation Samples")

with torch.no_grad():
    for index in tqdm(range(0, len(dataset), 16)):
        batch = dataset[index : index + 16]
        audio_arrays = [sample["array"] for sample in batch["audio_filepath"]]

        inputs = baseline_processor(audio_arrays, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
        logits = baseline_model(inputs.input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_ids = torch.argmax(logits, dim=-1)

        baseline_predictions.extend([transcription.lower() for transcription in predicted_transcriptions])
        baseline_references.extend([reference.lower() for reference in batch["text"]])

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

baseline_wer = wer_metric.compute(predictions=baseline_predictions, references=baseline_references)
baseline_cer = cer_metric.compute(predictions=baseline_predictions, references=baseline_references)

def calculate_classification_metrics(predictions, references):
    total_correct = 0
    total_predicted_words = 0
    total_reference_words = 0

    for prediction, reference in zip(predictions, references):
        prediction_words = set(prediction.split())
        reference_words = set(reference.split())

        total_correct += len(prediction_words & reference_words)
        total_predicted_words += len(prediction_words)
        total_reference_words += len(reference_words)

    precision = total_correct / total_predicted_words if total_predicted_words > 0 else 0
    recall = total_correct / total_reference_words if total_reference_words > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1_score

baseline_precision, baseline_recall, baseline_f1_score = calculate_classification_metrics(baseline_predictions, baseline_references)

print("")
print(f"The Word Error Rate (WER): {baseline_wer:.4f}")
print(f"The Character Error Rate (CER): {baseline_cer:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1-Score: {baseline_f1_score:.4f}")

In [ ]:
# Loading And Evaluating MMS-1B Speech Recognition Model Against The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

import gc
import torch
import evaluate

from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCTC, AutoProcessor

try:
    del baseline_model
    torch.cuda.empty_cache()
    gc.collect()
except NameError:
    pass

def collate_fn(batch):
    return {
        "audio": [sample["audio_filepath"]["array"] for sample in batch],
        "text": [sample["text"] for sample in batch]
    }

mms_model_identifier = "facebook/mms-1b-all"
print(f"Loading The MMS-1B (Massively Multilingual Speech) Model: {mms_model_identifier}")
print("")

mms_processor = AutoProcessor.from_pretrained(mms_model_identifier)
mms_model = AutoModelForCTC.from_pretrained(mms_model_identifier).to(device)
mms_processor.tokenizer.set_target_lang("eng")

try:
    mms_model.load_adapter("eng")
except:
    print("Default English Adapter Successfully Activated")

mms_model.eval()

mms_dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

mms_predictions = []
mms_references = []

print("Starting Parallel Inference On The MMS-1B (Massively Multilingual Speech) Model")

with torch.no_grad():
    for batch in tqdm(mms_dataloader):
        inputs = mms_processor(batch["audio"], sampling_rate=16000, return_tensors="pt", padding=True).to(device)
        logits = mms_model(inputs.input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_transcriptions = mms_processor.batch_decode(predicted_ids)

        mms_predictions.extend([transcription.lower() for transcription in predicted_transcriptions])
        mms_references.extend([reference.lower() for reference in batch["text"]])

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

mms_wer = wer_metric.compute(predictions=mms_predictions, references=mms_references)
mms_cer = cer_metric.compute(predictions=mms_predictions, references=mms_references)

def calculate_classification_metrics(predictions, references):
    total_correct = 0
    total_predicted_words = 0
    total_reference_words = 0

    for prediction, reference in zip(predictions, references):
        prediction_words = set(prediction.split())
        reference_words = set(reference.split())

        total_correct += len(prediction_words & reference_words)
        total_predicted_words += len(prediction_words)
        total_reference_words += len(reference_words)

    precision = total_correct / total_predicted_words if total_predicted_words > 0 else 0
    recall = total_correct / total_reference_words if total_reference_words > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1_score

mms_precision, mms_recall, mms_f1_score = calculate_classification_metrics(mms_predictions, mms_references)

print("")
print(f"The Word Error Rate (WER): {mms_wer:.4f}")
print(f"The Character Error Rate (CER): {mms_cer:.4f}")
print(f"Precision: {mms_precision:.4f}")
print(f"Recall: {mms_recall:.4f}")
print(f"F1-Score: {mms_f1_score:.4f}")

In [ ]:
# Loading And Evaluating The Distil-Whisper (Distilled Version Of Whisper) Model Against The Fine-Tuned Wav2Vec2 Model Using The Svarah Dataset

import gc
import re
import torch
import evaluate

from tqdm.auto import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

try:
    del mms_model
    torch.cuda.empty_cache()
    gc.collect()
except NameError:
    pass

distil_whisper_model_identifier = "distil-whisper/distil-large-v3"
print(f"Loading The Distil-Whisper (Distilled Version Of Whisper) Model: {distil_whisper_model_identifier}")
processor = AutoProcessor.from_pretrained(distil_whisper_model_identifier)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    distil_whisper_model_identifier,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True
).to(device)

speech_recognition_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    torch_dtype=torch.float16,
    device=0,
)
print("")
print("Starting The Inference On The Distil-Whisper (Distilled Version Of Whisper) Model")

audio_samples = [sample["array"] for sample in dataset["audio_filepath"]]
distil_whisper_results = speech_recognition_pipeline(audio_samples)

def clean_text(text):
    text = text.lower().strip()
    return re.sub(r"[^\w\s]", "", text)

distil_whisper_predictions = [clean_text(result["text"]) for result in distil_whisper_results]
distil_whisper_references = [clean_text(sample["text"]) for sample in dataset]

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

distil_whisper_wer = wer_metric.compute(predictions=distil_whisper_predictions, references=distil_whisper_references)
distil_whisper_cer = cer_metric.compute(predictions=distil_whisper_predictions, references=distil_whisper_references)

def calculate_classification_metrics(predictions, references):
    total_correct = 0
    total_predicted_words = 0
    total_reference_words = 0
    total_reference_words = 0

    for prediction, reference in zip(predictions, references):
        prediction_words = set(prediction.split())
        reference_words = set(reference.split())

        total_correct += len(prediction_words & reference_words)
        total_predicted_words += len(prediction_words)
        total_reference_words += len(reference_words)

    precision = total_correct / total_predicted_words if total_predicted_words > 0 else 0
    recall = total_correct / total_reference_words if total_reference_words > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

distil_whisper_precision, distil_whisper_recall, distil_whisper_f1_score = calculate_classification_metrics(
    distil_whisper_predictions,
    distil_whisper_references
)

print("")
print(f"The Word Error Rate (WER): {distil_whisper_wer:.4f}")
print(f"The Character Error Rate (CER: {distil_whisper_cer:.4f}")
print(f"Precision: {distil_whisper_precision:.4f}")
print(f"Recall: {distil_whisper_recall:.4f}")
print(f"F1-Score: {distil_whisper_f1_score:.4f}")

In [ ]:
# Creating A Comparative Evaluation Summary In A Tabular Format Across All Evaluated Speech Recognition Models

import pandas as pd

benchmarking_results = pd.DataFrame({
    "Model": [
        "Fine-Tuned Wav2Vec2",
        "Wav2Vec2 Base",
        "MMS-1B",
        "Distil-Whisper"
    ],
    "WER": [
        wer_score,
        baseline_wer,
        mms_wer,
        distil_whisper_wer
    ],
    "CER": [
        cer_score,
        baseline_cer,
        mms_cer,
        distil_whisper_cer
    ],
    "Precision": [
        precision,
        baseline_precision,
        mms_precision,
        distil_whisper_precision
    ],
    "Recall": [
        recall,
        baseline_recall,
        mms_recall,
        distil_whisper_recall
    ],
    "F1-Score": [
        f1_score,
        baseline_f1_score,
        mms_f1_score,
        distil_whisper_f1_score
    ]
})

benchmarking_results = benchmarking_results.round(4)

print("The Comparative Speech Recognition Evaluation Results In A Tabular Format")
print("")
display(benchmarking_results)